# Compressive Sensing Frameworks

This notebook implements and compares PCA and NN based frameworks for compressive sensing on PACE mission data. The frameworks explored include:

## Overview
- **Autoencoder (AE)**: Basic neural network autoencoder for feature extraction
- **Concrete Autoencoder (CAE)**: Autoencoder with concrete feature selection
- **Principal Component Analysis (PCA)**: Classic linear dimensionality reduction
- **Sparse PCA (CPPCA)**: PCA with sparsity constraints
- **Independent Component Analysis (ICA)**: Statistical technique for separating mixed signals
- **Feature Selection Autoencoder (FSAE)**: Autoencoder combined with Boruta feature selection
- **RPCA**: Robust PCA

## Purpose
This comparative analysis aims to identify the most effective techniques for compressive sensing applications in the PACE (Plankton, Aerosol, Cloud, ocean Ecosystem) mission data.

## Data
The notebook processes satellite observation data. More on the preprocessing in the Preprocessing_Granules notebook.

# 0. Installations and Imports

In [ ]:
import os
import shutil
from io import BytesIO

import pandas as pd
import boto3
import numpy as np
from dotenv import load_dotenv
import pyarrow.dataset as ds
import joblib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.cluster import MiniBatchKMeans
from boruta import BorutaPy
from tensorflow import keras
from tensorflow.keras import layers
import time
from sklearn.ensemble import ExtraTreesClassifier
from boruta import BorutaPy
from sklearn.decomposition import MiniBatchSparsePCA

from sklearn.model_selection import train_test_split
from sklearn.decomposition import DictionaryLearning

load_dotenv()

True

In [ ]:
def get_results(hist, autoencoder):
  # Validation loss
  val_loss = hist.history["val_loss"][-1]

  # Test loss
  X_reconstructed = autoencoder.predict(X_test.values)
  test_loss = float(np.mean(np.square(X_test.values - X_reconstructed)))

  # Per feature
  errors = np.square(X_test - X_reconstructed)
  feature_mse = np.mean(errors, axis=0)
  features_95_percentile = np.percentile(feature_mse, 95)

  # Per pixel
  errors = np.square(X_test - X_reconstructed)
  pixel_mse = np.mean(errors, axis=1)
  pixel_95_percentile = np.percentile(pixel_mse, 95)

  # Per Critical Wavelengths
  critical_wavelengths = [c for c in X.columns if c > 630 and c < 650]
  critical_wavelengths = [c for c in X.columns if c > 630 and c < 650]
  critical_wavelength_mse = feature_mse[critical_wavelengths]
  critical_99_percentile = np.percentile(critical_wavelength_mse, 99)


  performance = {
      "val_loss": val_loss,
      "test_loss": test_loss,
      "features_95_percentile": features_95_percentile,
      "pixel_95_percentile": pixel_95_percentile,
      "critical_wavelength_99_percentile": critical_99_percentile
  }

  return pd.Series(performance)

# 1. Import Data

In [ ]:
bucket = os.getenv("BUCKET_NAME")
results_bucket = os.getenv("RESULTS_BUCKET_NAME")
column_path = f"s3://{bucket}/headers.parquet"
columns =  pd.read_parquet(column_path)['0'].values

s3_folder = f"s3://{bucket}/samples/"
dataset = ds.dataset(s3_folder, format="parquet")
toscore = dataset.to_table().to_pandas()

toscore.columns = columns

bad_wavelengths = [c for c in toscore.columns if c < 320 or (c > 590 and c < 610)]

X = toscore.drop(columns=bad_wavelengths).dropna()
X

,320.169861,322.307007,324.506226,326.715118,328.912476,331.296478,334.044983,336.693512,338.931763,341.130859,...,894.614746,939.713013,1038.317017,1250.375000,1248.550049,1378.168945,1619.624023,1618.034058,2130.593018,2258.428955
13,0.608806,0.673659,0.761454,0.811532,0.851989,0.911868,0.974956,1.008217,1.008554,1.007460,...,0.730666,0.606328,0.746483,0.617481,0.656056,0.196888,0.538134,0.546682,0.484809,0.506884
14,0.622370,0.683895,0.767216,0.821973,0.866723,0.921690,0.978794,1.020782,1.020423,1.016808,...,0.772660,0.627736,0.808350,0.685337,0.685122,0.191732,0.574477,0.554163,0.485397,0.516773
15,0.623898,0.689095,0.778403,0.826054,0.872432,0.933025,0.990121,1.035906,1.042216,1.039792,...,0.892784,0.714389,0.915429,0.778097,0.803299,0.210163,0.653989,0.646882,0.562723,0.607542
16,0.626709,0.693794,0.788502,0.837849,0.880006,0.943437,1.002633,1.043321,1.046088,1.044327,...,0.918889,0.751358,0.931916,0.802266,0.825185,0.274438,0.650632,0.648142,0.562254,0.626396
17,0.617455,0.682618,0.776763,0.826803,0.870311,0.923386,0.983824,1.029886,1.031521,1.030544,...,0.877518,0.700519,0.872042,0.730413,0.783716,0.237287,0.569395,0.593536,0.501841,0.577241
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
367602,0.549988,0.648154,0.741501,0.814677,0.901940,0.988569,1.087735,1.171817,1.185125,1.189739,...,0.586918,0.396130,0.583895,0.487855,0.497741,0.014171,0.459525,0.455183,0.426592,0.400492
367603,0.548733,0.644926,0.741898,0.814960,0.894994,0.991513,1.093617,1.167367,1.182188,1.188912,...,0.579983,0.390870,0.578542,0.483582,0.494537,0.013229,0.455000,0.454177,0.447978,0.413710
367604,0.573637,0.639886,0.744978,0.811376,0.897219,0.997302,1.094799,1.173138,1.188374,1.189928,...,0.572872,0.385254,0.570327,0.475390,0.489545,0.013376,0.450772,0.451166,0.439275,0.406663
367605,0.553936,0.627581,0.741734,0.820987,0.897568,0.991305,1.093003,1.176127,1.192187,1.194019,...,0.546073,0.362146,0.539412,0.443473,0.465343,0.013425,0.419736,0.429272,0.414159,0.381129


In [ ]:
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

# Framework 1: Autoencoder (AE)

In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

input_dim = X_scaled.shape[1]
k = 32

input_layer = keras.Input(shape=(input_dim,))
encoded = layers.Dense(k, activation="relu")(input_layer)

decoded = layers.Dense(64, activation="relu")(encoded)
decoded = layers.Dense(128, activation="relu")(encoded)
decoded = layers.Dense(input_dim, activation="sigmoid")(decoded)

autoencoder = keras.Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer="adam", loss="mse")

hist = autoencoder.fit(
    X_scaled, X_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_split=0.2
)

X_reconstructed = autoencoder.predict(X_scaled)

In [ ]:
version = "v1"

performance = get_results(hist, autoencoder).to_frame()

validation_path = f"s3://{results_bucket}/auto-encoder/{version}/score.parquet"
weights_path = f"auto-encoder/{version}/ae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("artifacts", exist_ok=True)
autoencoder.save_weights("artifacts/ae_weights.weights.h5")

zip_path = shutil.make_archive("ae_savedmodel", "zip", "artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)

2156/2156 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step


# Framework 2: Concrete Autoencoder (CAE)


In [ ]:
class ConcreteSelect(layers.Layer):
    def __init__(self, k, input_dim, temperature=0.1, **kwargs):
        super().__init__(**kwargs)
        self.k = k
        self.input_dim = input_dim
        self.temperature = temperature

    def build(self, input_shape):
        self.logits = self.add_weight(
            shape=(self.k, self.input_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='logits'
        )

    def call(self, inputs, training=None):
        if training:
            uniform = tf.random.uniform(tf.shape(self.logits), minval=0, maxval=1)
            gumbel = -tf.math.log(-tf.math.log(uniform + 1e-20) + 1e-20)
            noisy_logits = (self.logits + gumbel) / self.temperature
            scores = tf.nn.softmax(noisy_logits, axis=-1)
        else:
            scores = tf.one_hot(tf.argmax(self.logits, axis=-1), depth=self.input_dim)
        return tf.matmul(inputs, tf.transpose(scores))

    def get_config(self):
        config = super().get_config()
        config.update({
            "k": self.k,
            "input_dim": self.input_dim,
            "temperature": self.temperature
        })
        return config


In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

input_dim = X_scaled.shape[1]
k = 32

input_layer = keras.Input(shape=(input_dim,))
encoded = ConcreteSelect(k=k, input_dim=input_dim)(input_layer)

decoded = layers.Dense(64, activation="relu")(encoded)
decoded = layers.Dense(128, activation="relu")(encoded)
decoded = layers.Dense(input_dim, activation="sigmoid")(decoded)

cae = keras.Model(inputs=input_layer, outputs=decoded)
cae.compile(optimizer="adam", loss="mse")

cae_hist = cae.fit(
    X_scaled, X_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_split=0.2
)

encoder = keras.Model(inputs=input_layer, outputs=encoded)
X_encoded = encoder.predict(X_scaled)

# Reconstruct
X_reconstructed = cae.predict(X_scaled)

In [ ]:
version = "v1"

performance = get_results(cae_hist, cae).to_frame()

validation_path = f"s3://{results_bucket}/concrete-auto-encoder/{version}/score.parquet"
weights_path = f"concrete-auto-encoder/{version}/cae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("cae_artifacts", exist_ok=True)
cae.save_weights("cae_artifacts/cae_weights.weights.h5")

zip_path = shutil.make_archive("cae_savedmodel", "zip", "artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

2156/2156 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step


,0
val_loss,0.000897
test_loss,0.001281
features_95_percentile,0.005744
pixel_95_percentile,0.000347
critical_wavelength_99_percentile,0.000276


# 4. PCA

With PCA, we had to add a layer of feature selection to the embeddings.


In [ ]:

spca = MiniBatchSparsePCA(
    n_components=8,
    alpha=20,
    batch_size=100,
    n_jobs=-1,
    max_iter=200,
    random_state=42
)
X_train_pred = spca.fit_transform(X_train)

nonzero_per_component = np.count_nonzero(spca.components_, axis=1)
print("Nonzero features per component:", nonzero_per_component)

used_features = set()
cum_used = []
for comp in spca.components_:
    used_features.update(np.where(comp != 0)[0])
    cum_used.append(len(used_features))

print("Cumulative unique features used:", cum_used)

Nonzero features per component: [204  81  67  28  10  80   4 113]
Cumulative unique features used: [204, 226, 228, 233, 243, 255, 256, 277]


In [ ]:
X_reconstructed = spca.inverse_transform(X_train_pred)
train_score = np.mean(np.square(X_train - X_reconstructed))
print("train_score: ", train_score)

X_test_pred = spca.inverse_transform(spca.transform(X_test))

test_score = np.mean(np.square(X_test - X_test_pred))
print("test_score: ", test_score)

train_score:  0.00020029703504275526
test_score:  0.00020515575202279


In [ ]:
val_loss = np.mean(np.square(X_train - X_reconstructed))

# Test loss
X_reconstructed = spca.inverse_transform(spca.transform(X_test))
test_loss = np.mean(np.square(X_test - X_test_pred))

# Per feature
errors = np.square(X_test - X_reconstructed)
feature_mse = np.mean(errors, axis=0)
features_95_percentile = np.percentile(feature_mse, 95)

# Per pixel
errors = np.square(X_test - X_reconstructed)
pixel_mse = np.mean(errors, axis=1)
pixel_95_percentile = np.percentile(pixel_mse, 95)

# Per Critical Wavelengths
critical_wavelengths = [c for c in X.columns if c > 630 and c < 650]
critical_wavelengths = [c for c in X.columns if c > 630 and c < 650]
critical_wavelength_mse = feature_mse[critical_wavelengths]
critical_99_percentile = np.percentile(critical_wavelength_mse, 99)


performance = {
    "val_loss": val_loss,
    "test_loss": test_loss,
    "features_95_percentile": features_95_percentile,
    "pixel_95_percentile": pixel_95_percentile,
    "critical_wavelength_99_percentile": critical_99_percentile
}

performance = pd.Series(performance)

performance

,0
val_loss,0.000200
test_loss,0.000205
features_95_percentile,0.000948
pixel_95_percentile,0.000736
critical_wavelength_99_percentile,0.000169


In [ ]:
version = "v1"

validation_path = f"s3://{results_bucket}/pca/{version}/score.parquet"

performance.to_frame().to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )


buffer = BytesIO()
joblib.dump(spca, buffer)
buffer.seek(0)

s3 = boto3.client('s3')
s3.upload_fileobj(buffer, results_bucket, f"pca/{version}/pca_model.joblib")

# 5. CPPCA

We will use a specific variation of CPPCA that is SparseRandomProjection

In [ ]:
from sklearn.random_projection import SparseRandomProjection
from sklearn.decomposition import PCA

In [ ]:
rp = SparseRandomProjection(n_components=32, random_state=42)
X_train_proj = rp.fit_transform(X_train)
X_test_proj = rp.transform(X_test)

In [ ]:
cppca_pca = PCA(n_components=32, random_state=42)
X_train_pred = cppca_pca.inverse_transform(cppca_pca.fit_transform(X_train_proj))
X_test_pred = cppca_pca.inverse_transform(cppca_pca.transform(X_test_proj))

In [ ]:
# Reconstruction
X_train_pred = rp.inverse_transform(X_train_pred)
X_test_pred = rp.inverse_transform(X_test_pred)

In [ ]:
# Train score
train_score = np.mean(np.square(X_train - X_train_pred))
print("train_score: ", train_score)

# Test score
test_score = np.mean(np.square(X_test - X_test_pred))
print("test_score: ", test_score)

train_score:  0.19198757
test_score:  0.1928409


In [ ]:
# Validation loss
val_loss = np.mean(np.square(X_train - X_train_pred))

# Test loss
test_loss = np.mean(np.square(X_test - X_test_pred))

# Per feature
errors = np.square(X_test - X_test_pred)
feature_mse = pd.Series(np.mean(errors, axis=0), index=X.columns)
features_95_percentile = np.percentile(feature_mse, 95)

# Per pixel
pixel_mse = np.mean(errors, axis=1)
pixel_95_percentile = np.percentile(pixel_mse, 95)

# Per critical wavelengths
critical_wavelengths = [c for c in X.columns if c > 630 and c < 650]
critical_wavelength_mse = feature_mse[critical_wavelengths]
critical_99_percentile = np.percentile(critical_wavelength_mse, 99)

performance = {
    "val_loss": val_loss,
    "test_loss": test_loss,
    "features_95_percentile": features_95_percentile,
    "pixel_95_percentile": pixel_95_percentile,
    "critical_wavelength_99_percentile": critical_99_percentile
}

performance = pd.Series(performance)
performance

,0
val_loss,0.191988
test_loss,0.192841
features_95_percentile,0.399908
pixel_95_percentile,0.568476
critical_wavelength_99_percentile,0.283641


In [ ]:
import numpy as np
from numpy.linalg import svd, eigh, pinv, norm

def cppca(X_train, X_test, K=50, L=10, J=5, max_iter=20, tol=1e-6, random_state=0):
    """
    K: projection dimension
    L: number of principal components to reconstruct
    J: number of projection partitions
    """

    np.random.seed(random_state)
    N, M = X_train.shape

    partitions = np.array_split(np.arange(M), J)
    projections, Y_proj, R_tilde, ritz_vectors = [], [], [], []

    for j, idx in enumerate(partitions):
        P = np.linalg.qr(np.random.randn(N, K))[0]
        projections.append(P)
        Xj = X_train[:, idx]
        Yj = P.T @ Xj
        Y_proj.append(Yj)

        Rj = (Yj @ Yj.T) / Yj.shape[1]
        eigvals, eigvecs = eigh(Rj)
        order = np.argsort(eigvals)[::-1]
        ritz_vectors.append(eigvecs[:, order[:L]])
        R_tilde.append(Rj)

    W_hat = np.zeros((N, L))
    for l in range(L):
        w = np.mean([P @ u[:, l] for P, u in zip(projections, ritz_vectors)], axis=0)
        w /= norm(w)

        for _ in range(max_iter):
            w_prev = w.copy()
            w_sum = np.zeros_like(w)
            for P, u in zip(projections, ritz_vectors):
                # build Q_j = P_perp + span(u)
                P_perp = np.eye(N) - P @ P.T
                Qj = np.hstack([P_perp, P @ u[:, [l]]])
                w_sum += Qj @ Qj.T @ w
            w = w_sum / J
            w /= norm(w)
            if norm(w - w_prev) < tol:
                break

        W_hat[:, l] = w

    X_recon_parts = []
    for P, Yj in zip(projections, Y_proj):
        X_hat_j = pinv(P.T @ W_hat) @ Yj
        X_recon_parts.append(W_hat @ X_hat_j)

    X_recon = np.concatenate(X_recon_parts, axis=1)

    coeff_test = W_hat.T @ X_test
    X_test_recon = W_hat @ coeff_test

    return W_hat, X_recon, X_test_recon

In [ ]:
from sklearn.model_selection import train_test_split

X_samp = X.sample(frac=0.001, random_state=42)
X_train, X_test = train_test_split(X_samp, test_size=0.2, random_state=42)

X_train = X_train.to_numpy().T
X_test  = X_test.to_numpy().T
X_train -= X_train.mean(axis=1, keepdims=True)
X_test  -= X_test.mean(axis=1, keepdims=True)

W_hat, X_train_pred, X_test_pred = cppca(X_train, X_test, K=50, L=15, J=10)


In [ ]:
# Validation loss
val_loss = np.mean(np.square(X_train - X_train_pred))

# Test loss
test_loss = np.mean(np.square(X_test - X_test_pred))

# Per feature
errors = np.square(X_test - X_test_pred).T
feature_mse = pd.Series(np.mean(errors, axis=0), index=X.columns)
features_95_percentile = np.percentile(feature_mse, 95)

# Per pixel
pixel_mse = np.mean(errors, axis=1)
pixel_95_percentile = np.percentile(pixel_mse, 95)

# Per critical wavelengths
critical_wavelengths = [c for c in X.columns if c > 630 and c < 650]
critical_wavelength_mse = feature_mse[critical_wavelengths]
critical_99_percentile = np.percentile(critical_wavelength_mse, 99)

performance = {
    "val_loss": val_loss,
    "test_loss": test_loss,
    "features_95_percentile": features_95_percentile,
    "pixel_95_percentile": pixel_95_percentile,
    "critical_wavelength_99_percentile": critical_99_percentile
}

performance = pd.Series(performance)
performance

,0
val_loss,0.022351
test_loss,0.015653
features_95_percentile,0.056901
pixel_95_percentile,0.039989
critical_wavelength_99_percentile,0.060906


In [ ]:
version = "v1"

validation_path = f"s3://{results_bucket}/cppca/{version}/score.parquet"

performance.to_frame().to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )


buffer = BytesIO()
joblib.dump(cppca, buffer)
buffer.seek(0)

s3 = boto3.client('s3')
s3.upload_fileobj(buffer, results_bucket, f"cppca/{version}/cppca_model.joblib")

# 6. ICA

In [ ]:
X = toscore.drop(columns=bad_wavelengths).dropna()

X_samp = X.sample(frac=0.01, random_state=42)
X_train, X_test = train_test_split(X_samp, test_size=0.2, random_state=42)

In [ ]:
n_components = 8
alpha = 0.8
random_state = 42

ica = DictionaryLearning(
    n_components=n_components,
    alpha=alpha,
    transform_algorithm='lasso_lars',
    random_state=random_state
)

S_train = ica.fit_transform(X_train)
S_test = ica.transform(X_test)
A_hat = ica.components_

X_train_pred = S_train @ A_hat
X_test_pred = S_test @ A_hat

In [ ]:
nonzero_counts = np.sum(np.abs(A_hat) > 1e-6, axis=1)
nonzero_counts

array([277, 277, 277, 277, 277, 277, 277, 277])

In [ ]:
# Validation loss
val_loss = np.mean(np.square(X_train - X_train_pred))

# Test loss
test_loss = np.mean(np.square(X_test - X_test_pred))

# Per feature
errors = np.square(X_test - X_test_pred)
feature_mse = pd.Series(np.mean(errors, axis=0), index=X.columns)
features_95_percentile = np.percentile(feature_mse, 95)

# Per pixel
pixel_mse = np.mean(errors, axis=1)
pixel_95_percentile = np.percentile(pixel_mse, 95)

# Per critical wavelengths
critical_wavelengths = [c for c in X.columns if c > 630 and c < 650]
critical_wavelength_mse = feature_mse[critical_wavelengths]
critical_99_percentile = np.percentile(critical_wavelength_mse, 99)

performance = {
    "val_loss": val_loss,
    "test_loss": test_loss,
    "features_95_percentile": features_95_percentile,
    "pixel_95_percentile": pixel_95_percentile,
    "critical_wavelength_99_percentile": critical_99_percentile
}

performance = pd.Series(performance)
performance

,0
val_loss,0.002689
test_loss,0.002696
features_95_percentile,0.007591
pixel_95_percentile,0.003381
critical_wavelength_99_percentile,0.001601


In [ ]:
version = "v1"

validation_path = f"s3://{results_bucket}/ica/{version}/score.parquet"

performance.to_frame().to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )


buffer = BytesIO()
joblib.dump(ica, buffer)
buffer.seek(0)

s3 = boto3.client('s3')
s3.upload_fileobj(buffer, results_bucket, f"ica/{version}/ica_model.joblib")

# 7. FSAE
Feature Selection based Auto Encoder

In [ ]:
X = toscore.drop(columns=bad_wavelengths).dropna()
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

In [ ]:
class ColumnSelector(layers.Layer):
    def __init__(self, indices, **kwargs):
        super().__init__(**kwargs)
        self.indices = tf.constant(indices, dtype=tf.int32)

    def call(self, inputs):
        return tf.gather(inputs, self.indices, axis=1)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"indices": self.indices.numpy().tolist()})
        return cfg


y_train = None
y = None

k = 32
feat_names = X.columns.tolist()

X_scaled = X_train.sample(frac=0.001).dropna().values

n_clusters = 10
km = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=256)
y_pseudo = km.fit_predict(X_scaled)

In [ ]:
est = ExtraTreesClassifier(
    n_estimators=300, n_jobs=-1, random_state=42, class_weight="balanced"
)

boruta = BorutaPy(
    estimator=est,
    n_estimators="auto",
    two_step=True,
    max_iter=40,
    random_state=42,
    verbose=0,
)

t0 = time.perf_counter()
boruta.fit(X_scaled, y_pseudo)
t1 = time.perf_counter()
print(f"Boruta took {(t1 - t0):.1f} seconds")

Boruta took 23.1 seconds; kept 270 features.


In [ ]:
confirmed_mask = boruta.support_
confirmed_idxs = np.where(confirmed_mask)[0]

if len(confirmed_idxs) < k:
    ranks = boruta.ranking_
    top_k_idxs = np.argsort(ranks)[:k]
    chosen_idxs = np.array(sorted(set(confirmed_idxs).union(set(top_k_idxs))))[:k]
else:
    ranks = boruta.ranking_
    confirmed_ranks = ranks[confirmed_idxs]
    order = np.argsort(confirmed_ranks)
    chosen_idxs = confirmed_idxs[order][:k]

chosen_idxs = np.sort(chosen_idxs)
selected_features = [feat_names[i] for i in chosen_idxs]

print(f"Selected {len(selected_features)} features (k={k}):")
print(selected_features)

Selected 32 features (k=32):
[326.7151184082031, 328.9124755859375, 331.2964782714844, 334.04498291015625, 336.6935119628906, 338.9317626953125, 341.130859375, 343.48248291015625, 345.90484619140625, 348.354736328125, 350.80511474609375, 353.2541198730469, 355.703857421875, 358.1629638671875, 360.615966796875, 831.984130859375, 834.4901123046875, 836.994873046875, 839.501220703125, 842.0137329101562, 844.521240234375, 847.038330078125, 849.5558471679688, 852.0602416992188, 854.56591796875, 857.0750732421875, 859.5819091796875, 862.0843505859375, 864.5924682617188, 867.0931396484375, 869.599609375, 872.112548828125]


In [ ]:
num_features = 277
X_scaled = X_train.dropna().values

input_dim = X_scaled.shape[1]
k = 32

inp = keras.Input(shape=(input_dim,))
encoded = ColumnSelector(indices=chosen_idxs, name="boruta_encoder")(inp)

x = layers.Dense(128, activation="relu")(encoded)
x = layers.Dense(256, activation="relu")(x)
out = layers.Dense(input_dim, activation="linear")(x)

fsae = keras.Model(inputs=inp, outputs=out, name="boruta_shallow_encoder_ae")
fsae.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

fsae_hist = fsae.fit(
    X_scaled,
    X_scaled,
    epochs=50,
    batch_size=64,
    shuffle=True,
    validation_split=0.2,
    verbose=1,
)

encoder = keras.Model(inputs=inp, outputs=encoded, name="boruta_encoder_only")

X_encoded = encoder.predict(X_scaled, batch_size=256)
X_reconstructed = fsae.predict(X_scaled, batch_size=256)


X_reconstructed = fsae.predict(X_scaled)

Epoch 1/50
3449/3449 ━━━━━━━━━━━━━━━━━━━━ 24s 6ms/step - loss: 0.0046 - val_loss: 8.8685e-04
Epoch 2/50
3449/3449 ━━━━━━━━━━━━━━━━━━━━ 39s 6ms/step - loss: 6.7489e-04 - val_loss: 5.3734e-04
Epoch 3/50
3449/3449 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 5.1469e-04 - val_loss: 5.3049e-04
Epoch 4/50
3449/3449 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 4.3565e-04 - val_loss: 4.0339e-04
Epoch 5/50
3449/3449 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - loss: 4.0458e-04 - val_loss: 4.9568e-04
Epoch 6/50
3449/3449 ━━━━━━━━━━━━━━━━━━━━ 21s 6ms/step - loss: 3.8164e-04 - val_loss: 3.5284e-04
Epoch 7/50
3449/3449 ━━━━━━━━━━━━━━━━━━━━ 26s 7ms/step - loss: 3.5952e-04 - val_loss: 3.9308e-04
Epoch 8/50
3449/3449 ━━━━━━━━━━━━━━━━━━━━ 32s 9ms/step - loss: 4.0167e-04 - val_loss: 3.3880e-04
Epoch 9/50
3449/3449 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 3.5509e-04 - val_loss: 2.7684e-04
Epoch 10/50
3449/3449 ━━━━━━━━━━━━━━━━━━━━ 50s 9ms/step - loss: 3.2426e-04 - val_loss: 3.3806e-04
Epoch 11/50
3449/3449 ━━━━━━━━━━━

In [ ]:
version = "v2"

performance = get_results(fsae_hist, fsae).to_frame()

validation_path = f"s3://{results_bucket}/fsae/{version}/score.parquet"
weights_path = f"fsae/{version}/fsae_savedmodel.zip"

performance.to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )

os.makedirs("fsae_artifacts", exist_ok=True)
fsae.save_weights("fsae_artifacts/fsae_weights.weights.h5")

zip_path = shutil.make_archive("fsae_savedmodel", "zip", "artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, results_bucket, weights_path)


performance

2156/2156 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step


,0
val_loss,0.000221
test_loss,0.000217
features_95_percentile,0.000683
pixel_95_percentile,0.000844
critical_wavelength_99_percentile,0.000194


# 8 Robust PCA (RPCA) and Incremental Robust PCA (IRPCA)

In [ ]:
def robust_pca(X, rank=50, lam=None, mu=None, max_iter=50, tol=1e-7):
    X = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
    m, n = X.shape
    if lam is None:
        lam = 1 / np.sqrt(max(m, n))
    if mu is None:
        mu = (m * n) / (4.0 * np.sum(np.abs(X)))

    L = np.zeros_like(X)
    S = np.zeros_like(X)
    Y = np.zeros_like(X)

    for i in range(max_iter):
        # 1. Low-rank update via truncated SVD (keep only top `rank`)
        U, sigma, Vt = np.linalg.svd(X - S + (1/mu) * Y, full_matrices=False)
        sigma_thresh = np.maximum(sigma - 1/mu, 0)
        U_r, sigma_r, Vt_r = U[:, :rank], sigma_thresh[:rank], Vt[:rank, :]
        L = (U_r * sigma_r) @ Vt_r

        # 2. Sparse update
        S = np.sign(X - L + (1/mu) * Y) * np.maximum(np.abs(X - L + (1/mu) * Y) - lam/mu, 0)

        # 3. Dual variable update
        Z = X - L - S
        Y = Y + mu * Z

        err = np.linalg.norm(Z, 'fro') / np.linalg.norm(X, 'fro')
        if i % 5 == 0:
            print(f"iter {i}: error = {err:.2e}")
        if err < tol:
            break

    return L, S

In [ ]:
k = 8

X = toscore.drop(columns=bad_wavelengths).dropna()
X_samp = X.sample(frac=0.01, random_state=42)
X_train, X_test = train_test_split(X_samp, test_size=0.2, random_state=42)

L, S = robust_pca(X_train, rank=k, max_iter=50)
X_train_pred = L

U, sigma, Vt = np.linalg.svd(L, full_matrices=False)

iter 0: error = 1.51e-02
iter 5: error = 9.24e-03
iter 10: error = 8.29e-03
iter 15: error = 9.17e-03
iter 20: error = 7.77e-03
iter 25: error = 8.29e-03
iter 30: error = 8.20e-03
iter 35: error = 8.45e-03
iter 40: error = 8.28e-03
iter 45: error = 7.10e-03
train_score: 0.0000889
test_score:  0.0000602
effective rank (forced): 8


In [ ]:
k = 4
V = Vt.T[:, :k]

X_train_pred = X_train.to_numpy() @ V @ V.T
X_test_pred = X_test.to_numpy() @ V @ V.T

train_score = np.mean((X_train.to_numpy() - X_train_pred) ** 2)
test_score = np.mean((X_test.to_numpy() - X_test_pred) ** 2)

print(f"train_score: {train_score:.7f}")
print(f"test_score:  {test_score:.7f}")
print(f"effective rank (forced): {V.shape[1]}")


train_score: 0.0004221
test_score:  0.0004172
effective rank (forced): 4


In [ ]:
# Validation loss
val_loss = np.mean(np.square(X_train - X_train_pred))

# Test loss
test_loss = np.mean(np.square(X_test - X_test_pred))

# Per feature
errors = np.square(X_test - X_test_pred)
feature_mse = pd.Series(np.mean(errors, axis=0), index=X.columns)
features_95_percentile = np.percentile(feature_mse, 95)

# Per pixel
pixel_mse = np.mean(errors, axis=1)
pixel_95_percentile = np.percentile(pixel_mse, 95)

# Per critical wavelengths
critical_wavelengths = [c for c in X.columns if c > 630 and c < 650]
critical_wavelength_mse = feature_mse[critical_wavelengths]
critical_99_percentile = np.percentile(critical_wavelength_mse, 99)

performance = {
    "val_loss": val_loss,
    "test_loss": test_loss,
    "features_95_percentile": features_95_percentile,
    "pixel_95_percentile": pixel_95_percentile,
    "critical_wavelength_99_percentile": critical_99_percentile
}

performance = pd.Series(performance)
performance

,0
val_loss,0.000422
test_loss,0.000417
features_95_percentile,0.001293
pixel_95_percentile,0.001567
critical_wavelength_99_percentile,0.000599


In [ ]:
version = "v1"

validation_path = f"s3://{results_bucket}/rpca/{version}/score.parquet"

performance.to_frame().to_parquet(
      validation_path,
      index=True,
      engine="pyarrow",
  )


buffer = BytesIO()
joblib.dump(robust_pca, buffer)
buffer.seek(0)

s3 = boto3.client('s3')
s3.upload_fileobj(buffer, results_bucket, f"rpca/{version}/rpca_model.joblib")